# EDM–Fuzzy Entropy + Fault Injection (4 humidity sensors)
Pipeline sesuai diagram: baseline → fault injection per sensor → hitung EDM–Fuzzy Entropy per sensor (vector per skala) → gabung fitur → ANN + grid search.

**Catatan percepatan**: notebook ini **tanpa downsampling dan tanpa windowing**; fitur dihitung dari full series per skenario. Percepatan yang digunakan tetap dari sampling Monte Carlo pada perhitungan similarity untuk menekan kompleksitas $O(N^2)$.


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import time, tracemalloc, logging
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score, accuracy_score


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# === Sampling controls (no downsampling/windowing) ===
USE_BALANCED_SUBSAMPLE = True
USE_RANDOM_SAMPLE_SUBSAMPLE = False  # if True, randomly sample samples (extra reduction)

MAX_PER_CLASS = 500  # ignored if USE_BALANCED_SUBSAMPLE=False
MAX_SAMPLES_TOTAL = 4000  # used if USE_RANDOM_SAMPLE_SUBSAMPLE=True
RANDOM_SEED = 42


# Load Data (4 Sensor)

**Tujuan:** memuat time-series kelembaban, memilih **4 kolom numerik** sebagai sensor, lalu membersihkan nilai hilang.

**Input**
- File CSV `tabel_sensor4_generated.csv` (diunduh via URL GitHub).
- Terdapat kolom "kelembaban1","kelembaban2","kelembaban3","kelembaban4"

**Proses di kode**
- `load_default_data()` mengunduh CSV → `pd.read_csv(...)` → DataFrame `df`.
- `cols = ["kelembaban1","kelembaban2","kelembaban3","kelembaban4"]`
- `X_raw = df[cols].to_numpy(dtype=float)`
- Imputasi NaN berurutan: `ffill → bfill → median` per kolom.

**Output**
- `cols`: daftar nama 4 kolom sensor yang dipakai.
- `X`: `numpy.ndarray` bentuk `(T, 4)` berisi sinyal sensor yang sudah bebas NaN.


## 1) Load data (4 sensor)
File: `tabel_sensor4_generated.csv` (diasumsikan berisi 4 kolom sensor kelembaban).

In [ ]:
import pandas as pd
import requests

def load_default_data():
    url = "https://raw.githubusercontent.com/vousmeevoyez/public-files/refs/heads/main/tabel_sensor4_generated.csv"
    response = requests.get(url)
    response.raise_for_status()
    from io import StringIO
    return pd.read_csv(StringIO(response.text))
df = load_default_data()
df.head(), df.shape


In [ ]:
# pilih 4 kolom
cols = ["kelembaban1","kelembaban2","kelembaban3","kelembaban4"]
X_raw = df[cols].to_numpy(dtype=float)

# imputasi NaN sederhana (ffill → bfill → median)
X = pd.DataFrame(X_raw, columns=cols)
X = X.fillna(method="ffill").fillna(method="bfill").fillna(X.median(numeric_only=True))
X = X.to_numpy()

print("Columns:", cols)
print("Shape:", X.shape)


# Sampling & Reduksi Data (tanpa downsampling/windowing)

**Tujuan:** menjaga seluruh time-series utuh dan hanya melakukan reduksi jika diperlukan di level sampel.

- Tidak ada **downsampling**.
- Tidak ada **windowing**; tiap skenario menghasilkan satu sampel full series.
- **Subsampling** opsional untuk membatasi jumlah sampel per kelas (jika kelas sangat banyak).


## 2) Data sampling & reduction
Untuk mempercepat (opsional):
- **Subsampling** per kelas (opsional)
- **Random sampling** total sampel (opsional)


In [ ]:
rng = np.random.default_rng(RANDOM_SEED)

# No downsampling/windowing: use full series as one sample
X_ds = X.copy()
W_base = X_ds[None, :, :]

print("After sampling setup:")
print("  X:", X_ds.shape)
print("  Samples:", W_base.shape)


# Fault Injection (per Sensor)

**Tujuan:** membangkitkan data *faulty* dari baseline dengan menyisipkan fault **per sensor** sesuai skenario.

## A. Simulator fault 1D (per sensor)
Semua simulator menerima **sinyal 1D** `x` bentuk `(T,)` dan menghasilkan:
- `y`: sinyal setelah fault `(T,)`
- `m`: mask boolean lokasi fault `(T,)`

- `simulate_drift_fault`: menambah drift linier `t*intensity`.
- `simulate_spike_fault`: menambah spike periodik (besar spike ∝ `std(x)`).
- `simulate_bias_fault`: menambah offset konstan (bias).
- `simulate_hardware_fault`: kombinasi **stuck** (nilai diganti nilai lain) dan **loss** (NaN).

## B. Multi-fault dalam satu sensor
`simulate_multiple_faults(x, faults)` menerapkan daftar fault berurutan; mask digabung `OR` sehingga menandai titik yang terkena salah satu fault.

## C. Injeksi ke 4 sensor
`inject_faults_multisensor(X, scenario_faults)`:
- loop ke-4 sensor: injeksi skenario yang sama, tetapi seed berbeda per sensor
- menangani NaN (loss) dengan `ffill → bfill → median`

**Output:**  
- `Y`: `(T,4)` data setelah fault  
- `M`: `(T,4)` mask fault per sensor

## D. Label per skenario
Pada versi ini tidak ada windowing; setiap skenario menjadi satu sampel full series.


## 3) Fault injection (sesuai skenario)
Fault disisipkan **per sensor**. Skenario minimal 6 kombinasi 2-fault.

Output: dataset sampel berlabel (kelas) per skenario.


In [ ]:

# --- Fault simulators (subtler, different seed per call) ---
def simulate_drift_fault(x, intensity=0.02, seed=None):
    rng=np.random.default_rng(seed)
    alpha = intensity
    drift = np.arange(len(x)) * alpha
    y=x+drift; m=np.abs(drift)>1e-6; return y,m

def simulate_spike_fault(x, intensity=0.08, p=0.015, seed=None):
    rng=np.random.default_rng(seed)
    tau = max(1, int(1.0 / p)) if p > 0 else len(x)
    spikes = (np.arange(len(x)) % tau == 0).astype(float) * (intensity * np.nanstd(x))
    y=x+spikes; m=spikes!=0; return y,m

def simulate_bias_fault(x, bias=0.08, seed=None):
    y=x+bias; m=np.ones(len(x),bool); return y,m

def simulate_hardware_fault(x, stuck_prob=0.08, loss_prob=0.05, seed=None):
    rng=np.random.default_rng(seed)
    n = len(x)
    rand_vals = rng.random(n)
    idx=rng.integers(n, size=n)
    m1 = rand_vals < stuck_prob
    y = x.copy()
    y[m1] = x[idx[m1]]
    m2 = rand_vals < loss_prob
    y[m2] = np.nan
    return y, (m1 | m2)

def simulate_multiple_faults(x, faults, seed=None):
    y=x.copy(); m=np.zeros(len(x),bool)
    for f,kw in faults:
        y,mi=f(y,**kw,seed=seed); m|=mi
    return y,m


def simulate_choose_one(x, options, seed=None):
    rng = np.random.default_rng(seed)
    f, kw = options[rng.integers(len(options))]
    return f(x, **kw, seed=seed)

# fault scenario ditambah 2,3,4 fault
SCENARIOS = {
    # 1
    "faulty": [(
        simulate_choose_one, {
            "options": [
                (simulate_drift_fault, {"intensity":0.02}),
                (simulate_spike_fault, {"intensity":0.08, "p":0.015}),
                (simulate_bias_fault, {"bias":0.08}),
                (simulate_hardware_fault, {"stuck_prob":0.08, "loss_prob":0.05}),
            ]
        }
    )],

    # 2
    "drift": [(simulate_drift_fault, {"intensity":0.02})],
    "spike": [(simulate_spike_fault, {"intensity":0.08, "p":0.015})],
    "bias": [(simulate_bias_fault, {"bias":0.08})],
    "hardware": [(simulate_hardware_fault, {"stuck_prob":0.08, "loss_prob":0.05})],

    # 3
    "bias+malfunc": [(simulate_bias_fault, {"bias":0.08}),
                     (simulate_hardware_fault, {"stuck_prob":0.08, "loss_prob":0.05})],
    "spike+malfunc": [(simulate_spike_fault, {"intensity":0.08, "p":0.015}),
                      (simulate_hardware_fault, {"stuck_prob":0.08, "loss_prob":0.05})],
    "spike+bias": [(simulate_spike_fault, {"intensity":0.08, "p":0.015}),
                   (simulate_bias_fault, {"bias":0.08})],
    "drift+malfunc": [(simulate_drift_fault, {"intensity":0.02}),
                      (simulate_hardware_fault, {"stuck_prob":0.08, "loss_prob":0.05})],
    "drift+bias": [(simulate_drift_fault, {"intensity":0.02}),
                   (simulate_bias_fault, {"bias":0.08})],
    "drift+spike": [(simulate_drift_fault, {"intensity":0.02}),
                    (simulate_spike_fault, {"intensity":0.08, "p":0.015})],

    # 4
    "spike+bias+malfunc": [
        (simulate_spike_fault, {"intensity":0.08, "p":0.015}),
        (simulate_bias_fault, {"bias":0.08}),
        (simulate_hardware_fault, {"stuck_prob":0.08, "loss_prob":0.05})
    ],
    "drift+bias+malfunc": [
        (simulate_drift_fault, {"intensity":0.02}),
        (simulate_bias_fault, {"bias":0.08}),
        (simulate_hardware_fault, {"stuck_prob":0.08, "loss_prob":0.05})
    ],
    "spike+drift+malfunc": [
        (simulate_spike_fault, {"intensity":0.08, "p":0.015}),
        (simulate_drift_fault, {"intensity":0.02}),
        (simulate_hardware_fault, {"stuck_prob":0.08, "loss_prob":0.05})
    ],
    "drift+spike+bias": [
        (simulate_drift_fault, {"intensity":0.02}),
        (simulate_spike_fault, {"intensity":0.08, "p":0.015}),
        (simulate_bias_fault, {"bias":0.08}),
    ],

    # 5
    "spike+bias+malfunc+drift": [
        (simulate_spike_fault, {"intensity":0.08, "p":0.015}),
        (simulate_bias_fault, {"bias":0.08}),
        (simulate_hardware_fault, {"stuck_prob":0.08, "loss_prob":0.05}),
        (simulate_drift_fault, {"intensity":0.02}),
    ],
}


In [ ]:
def inject_faults_multisensor(X, scenario_faults, seed=0):
    # X: (T,4) -> Y: (T,4), M: (T,4) boolean mask
    rng = np.random.default_rng(seed)
    Y = X.copy()
    M = np.zeros_like(Y, dtype=bool)
    for s in range(Y.shape[1]):
        y, m = simulate_multiple_faults(Y[:,s], scenario_faults, seed=int(rng.integers(1e9)))
        Y[:,s] = y
        M[:,s] = m
    # replace NaN (from malfunc loss) with forward fill then median per sensor
    Ydf = pd.DataFrame(Y)
    Ydf = Ydf.fillna(method="ffill").fillna(method="bfill").fillna(Ydf.median(numeric_only=True))
    return Ydf.to_numpy(), M

# Build dataset across scenarios + normal (no windowing)
datasets = []
labels = []
scenario_names = ["normal"] + list(SCENARIOS.keys())

# normal
W0 = X_ds[None, :, :]
datasets.append(W0); labels.append(np.zeros(len(W0), dtype=int))

# scenarios
for k, (name, faults) in enumerate(SCENARIOS.items(), start=1):
    Y, M = inject_faults_multisensor(X_ds, faults, seed=100+k)
    Wk = Y[None, :, :]
    datasets.append(Wk); labels.append(np.full(len(Wk), k, dtype=int))
    print(name, "samples:", len(Wk))

W_all = np.concatenate(datasets, axis=0)
y_all = np.concatenate(labels, axis=0)
print("Total samples:", W_all.shape, "Classes:", np.unique(y_all, return_counts=True))


### Optional: balanced sampling per class (reduction)
Jika data besar, batasi `max_per_class` untuk mempercepat perhitungan entropy dan ANN.

In [ ]:
def balanced_subsample(Xw, y, max_per_class=600, seed=0):
    rng=np.random.default_rng(seed)
    keep=[]
    for c in np.unique(y):
        idx=np.where(y==c)[0]
        if len(idx)>max_per_class:
            idx=rng.choice(idx, size=max_per_class, replace=False)
        keep.append(idx)
    keep=np.concatenate(keep)
    rng.shuffle(keep)
    return Xw[keep], y[keep]

def random_sample_subsample(Xw, y, max_total=4000, seed=0):
    if len(Xw) <= max_total:
        return Xw, y
    rng=np.random.default_rng(seed)
    idx=rng.choice(np.arange(len(Xw)), size=max_total, replace=False)
    return Xw[idx], y[idx]

W_s, y_s = W_all, y_all

if USE_BALANCED_SUBSAMPLE:
    W_s, y_s = balanced_subsample(W_s, y_s, max_per_class=MAX_PER_CLASS, seed=RANDOM_SEED)

if USE_RANDOM_SAMPLE_SUBSAMPLE:
    W_s, y_s = random_sample_subsample(W_s, y_s, max_total=MAX_SAMPLES_TOTAL, seed=RANDOM_SEED)

print("After subsample toggles:", W_s.shape, np.unique(y_s, return_counts=True))


# EDM–Fuzzy Entropy (Fitur Utama) + Metode Pembanding

**Tujuan:** untuk setiap sampel (full series) & setiap sensor, menghitung **vektor entropy multi-skala**
 dengan beberapa metode untuk perbandingan: **EDM–Fuzzy, CMSE, FME, dan JSD–Fuzzy**.

## Langkah per skala `s` (EDM–Fuzzy)
1) **Coarse-graining** (`coarse_grain_mean`):  
   - Input: `x` (sinyal 1D), `s`  
   - Output: `y` yang diperkasar (panjang ~ `T/s`) dengan rata-rata blok.

2) **Embedding** (`embed_matrix`):  
   - Input: `y`, dimensi `m`  
   - Output: matriks `V_m` bentuk `(Nemb, m)` (sliding window).

3) **Jarak Euclidean & Similarity fuzzy** (`fuzzy_phi`):  
   - Menghitung `phi_m` sebagai rata-rata similarity fuzzy:  
     
     \[
     \mu(d)=
     \frac{1}{1+(d/r)^2}
     \]
   - **Percepatan:** sampling `n_ref` embedding sebagai referensi (Monte Carlo),
     sehingga tidak perlu semua pasangan (mengurangi beban O(N²)).

4) **Entropy** (`edm_fuzzy_entropy_1d`):  
   - Hitung `phi_m` dan `phi_{m+1}` →  
     \[
     E(s)=\ln\left(
     \frac{\phi_m}{\phi_{m+1}}\right)
     \]

**Output:** untuk satu sensor pada satu sampel → vektor `E` bentuk `(S,)`.

Catatan: untuk metode lain (CMSE, FME, JSD–Fuzzy), alur tetap mengikuti skala `s`
dengan definisi entropy masing-masing.


## 4) Entropy Multi-Method (EDM–Fuzzy, CMSE, FME, JSD–Fuzzy)
Implementasi ringkas mengikuti blok diagram:
- EDM–Fuzzy: seperti definisi di atas (jarak Euclidean + similarity fuzzy).
- CMSE: Composite Multiscale Sample Entropy (SampEn).
- FME: Fuzzy Multiscale Entropy (jarak Chebyshev + similarity fuzzy).
- JSD–Fuzzy: Jensen–Shannon divergence antar distribusi similarity fuzzy (m vs m+1).

**Percepatan:** gunakan sampling indeks embedding (Monte Carlo) sehingga tidak perlu semua pasangan vektor.

**Catatan:** pipeline di bawah kini menghitung semua metode **secara paralel** untuk perbandingan.


In [ ]:
def coarse_grain_mean(x, s):
    n = (len(x)//s)*s
    if n <= 0:
        return np.array([], dtype=float)
    xs = x[:n].reshape(-1, s).mean(axis=1)
    return xs

def coarse_grain_multi(x, s):
    # Composite coarse-graining dengan offset 0..s-1 (CMSE)
    ys = []
    for k in range(s):
        n = (len(x)-k)//s
        if n <= 0:
            continue
        y = x[k:k+n*s].reshape(-1, s).mean(axis=1)
        if len(y) > 0:
            ys.append(y)
    return ys

def embed_matrix(y, m):
    # y: (L,) -> (L-m+1, m)
    L = len(y)
    if L < m:
        return np.empty((0, m), dtype=float)
    return np.lib.stride_tricks.sliding_window_view(y, m)

def fuzzy_phi(V, r, n_ref=256, seed=0):
    # V: (N,m). Approximate phi by sampling reference vectors i
    # phi = average over i of (1/(N-1) sum_{j!=i} mu(d(i,j)))
    rng=np.random.default_rng(seed)
    N = V.shape[0]
    if N < 3:
        return np.nan
    idx = np.arange(N)
    if N > n_ref:
        ref = rng.choice(idx, size=n_ref, replace=False)
    else:
        ref = idx
    # compute distances from each ref to all vectors: (n_ref,N)
    # squared euclidean: ||a-b||^2 = a^2 + b^2 - 2ab
    A = V[ref]
    a2 = np.sum(A*A, axis=1, keepdims=True)
    b2 = np.sum(V*V, axis=1, keepdims=True).T
    d2 = a2 + b2 - 2*(A @ V.T)
    d2 = np.maximum(d2, 0.0)
    d = np.sqrt(d2)
    mu = 1.0/(1.0 + (d/(r+1e-12))**2)
    # exclude self similarity where i==j (only when ref contains i)
    for ri, i in enumerate(ref):
        mu[ri, i] = 0.0
    Bi = mu.sum(axis=1) / (N-1)
    return Bi.mean()

def fuzzy_phi_cheb(V, r, n_ref=256, seed=0):
    # Similarity fuzzy dengan jarak Chebyshev (FME)
    rng=np.random.default_rng(seed)
    N = V.shape[0]
    if N < 3:
        return np.nan
    idx = np.arange(N)
    if N > n_ref:
        ref = rng.choice(idx, size=n_ref, replace=False)
    else:
        ref = idx
    A = V[ref]
    d = np.max(np.abs(A[:, None, :] - V[None, :, :]), axis=2)
    mu = 1.0/(1.0 + (d/(r+1e-12))**2)
    for ri, i in enumerate(ref):
        mu[ri, i] = 0.0
    Bi = mu.sum(axis=1) / (N-1)
    return Bi.mean()

def sample_entropy_1d(y, m, r):
    # SampEn dasar untuk CMSE (pakai jarak Chebyshev)
    V_m  = embed_matrix(y, m)
    V_m1 = embed_matrix(y, m+1)

    def _count_similar(V):
        N = V.shape[0]
        if N < 2:
            return 0, 0
        count = 0
        total = 0
        for i in range(N-1):
            d = np.max(np.abs(V[i+1:] - V[i]), axis=1)
            count += np.sum(d <= r)
            total += (N - i - 1)
        return count, total

    c_m, t_m = _count_similar(V_m)
    c_m1, t_m1 = _count_similar(V_m1)
    if t_m == 0 or t_m1 == 0 or c_m == 0 or c_m1 == 0:
        return np.nan
    return -np.log((c_m1 / t_m1) / (c_m / t_m))

def edm_fuzzy_entropy_1d(x, scales, m=2, r_ratio=0.2, n_ref=256, seed=0):
    # returns entropy vector [len(scales)]
    out=[]
    for s in scales:
        y = coarse_grain_mean(x, s)
        if len(y) < (m+2):
            out.append(np.nan); continue
        r = r_ratio * np.std(y, ddof=1)
        V_m  = embed_matrix(y, m)
        V_m1 = embed_matrix(y, m+1)
        phi_m  = fuzzy_phi(V_m,  r, n_ref=n_ref, seed=seed+11*s)
        phi_m1 = fuzzy_phi(V_m1, r, n_ref=n_ref, seed=seed+17*s)
        if (phi_m is None) or (phi_m1 is None) or (phi_m<=0) or (phi_m1<=0) or np.isnan(phi_m) or np.isnan(phi_m1):
            out.append(np.nan)
        else:
            out.append(np.log(phi_m/phi_m1))
    return np.array(out, dtype=float)

def cmse_1d(x, scales, m=2, r_ratio=0.2):
    # CMSE: Composite Multiscale Sample Entropy
    out = []
    for s in scales:
        ys = coarse_grain_multi(x, s)
        if not ys:
            out.append(np.nan); continue
        ent_list = []
        for y in ys:
            if len(y) < (m+2):
                continue
            r = r_ratio * np.std(y, ddof=1)
            ent_list.append(sample_entropy_1d(y, m=m, r=r))
        if len(ent_list) == 0:
            out.append(np.nan)
        else:
            out.append(np.nanmean(ent_list))
    return np.array(out, dtype=float)

def fme_1d(x, scales, m=2, r_ratio=0.2, n_ref=256, seed=0):
    # FME: Fuzzy Multiscale Entropy (jarak Chebyshev)
    out=[]
    for s in scales:
        y = coarse_grain_mean(x, s)
        if len(y) < (m+2):
            out.append(np.nan); continue
        r = r_ratio * np.std(y, ddof=1)
        V_m  = embed_matrix(y, m)
        V_m1 = embed_matrix(y, m+1)
        phi_m  = fuzzy_phi_cheb(V_m,  r, n_ref=n_ref, seed=seed+11*s)
        phi_m1 = fuzzy_phi_cheb(V_m1, r, n_ref=n_ref, seed=seed+17*s)
        if (phi_m is None) or (phi_m1 is None) or (phi_m<=0) or (phi_m1<=0) or np.isnan(phi_m) or np.isnan(phi_m1):
            out.append(np.nan)
        else:
            out.append(np.log(phi_m/phi_m1))
    return np.array(out, dtype=float)

def fuzzy_similarity_samples(V, r, n_ref=256, seed=0):
    # Ambil sampel nilai similarity fuzzy (untuk JSD)
    rng=np.random.default_rng(seed)
    N = V.shape[0]
    if N < 3:
        return np.array([], dtype=float)
    idx = np.arange(N)
    if N > n_ref:
        ref = rng.choice(idx, size=n_ref, replace=False)
    else:
        ref = idx
    A = V[ref]
    a2 = np.sum(A*A, axis=1, keepdims=True)
    b2 = np.sum(V*V, axis=1, keepdims=True).T
    d2 = a2 + b2 - 2*(A @ V.T)
    d2 = np.maximum(d2, 0.0)
    d = np.sqrt(d2)
    mu = 1.0/(1.0 + (d/(r+1e-12))**2)
    for ri, i in enumerate(ref):
        mu[ri, i] = np.nan
    return mu[~np.isnan(mu)].ravel()

def jsd_fuzzy_entropy_1d(x, scales, m=2, r_ratio=0.2, n_ref=256, seed=0, bins=20):
    # JSD-Fuzzy: JSD antara distribusi similarity fuzzy (m vs m+1)
    out=[]
    bin_edges = np.linspace(0.0, 1.0, bins+1)
    for s in scales:
        y = coarse_grain_mean(x, s)
        if len(y) < (m+2):
            out.append(np.nan); continue
        r = r_ratio * np.std(y, ddof=1)
        V_m  = embed_matrix(y, m)
        V_m1 = embed_matrix(y, m+1)
        mu_m  = fuzzy_similarity_samples(V_m,  r, n_ref=n_ref, seed=seed+11*s)
        mu_m1 = fuzzy_similarity_samples(V_m1, r, n_ref=n_ref, seed=seed+17*s)
        if (len(mu_m) == 0) or (len(mu_m1) == 0):
            out.append(np.nan); continue
        p,_ = np.histogram(mu_m,  bins=bin_edges)
        q,_ = np.histogram(mu_m1, bins=bin_edges)
        p = p.astype(float); q = q.astype(float)
        if p.sum() == 0 or q.sum() == 0:
            out.append(np.nan); continue
        p /= p.sum(); q /= q.sum()
        m_ = 0.5 * (p + q)
        eps = 1e-12
        kl_p = np.sum(p * np.log((p + eps) / (m_ + eps)))
        kl_q = np.sum(q * np.log((q + eps) / (m_ + eps)))
        jsd = 0.5 * (kl_p + kl_q)
        out.append(jsd)
    return np.array(out, dtype=float)

# quick sanity check on one sample/sensor
scales = np.arange(1, 11)
e = edm_fuzzy_entropy_1d(W_s[0,:,0], scales=scales, m=2, r_ratio=0.2, n_ref=256, seed=0)
e


# Fitur Akhir: Konkatenasi 4 Sensor (Multi-Method)

`compute_features_entropy(W, scales, method=...)`

**Input**
- `W`: `(Nsamp, T, 4)` kumpulan sampel full series
- `scales`: `1..S`
- `method`: `EDM-Fuzzy`, `CMSE`, `FME`, `JSD-Fuzzy`

**Proses**
- untuk tiap sampel dan tiap sensor:
  - hitung entropy `(S,)`
- fitur akhir per sampel = konkatenasi 4 sensor `(4*S,)`
- *semua metode dihitung* → hasil disimpan di `F_by_method` untuk perbandingan

**Output**
- `F_by_method`: dict fitur `(Nsamp, 4*S)` untuk **semua metode**
- `F`: fitur default dari `default_method` (dipakai jika `show_all_methods = False`)
- NaN (jika ada) diimputasi median per kolom


## 5) Hitung entropy per sensor → gabung fitur
Entropy dihitung per sensor menghasilkan vector:
- Sensor1: $E_1(1..S)$
- Sensor2: $E_2(1..S)$
- Sensor3: $E_3(1..S)$
- Sensor4: $E_4(1..S)$

Fitur akhir = konkatenasi $[E_1,E_2,E_3,E_4]$ ukuran `4*S`.

Catatan A: uji beberapa pilihan skala untuk CV (kestabilan entropy).
Catatan B: semua metode dihitung **bersamaan** agar perbandingan mudah.


In [ ]:
def compute_features_entropy(W, scales, method="EDM-Fuzzy", m=2, r_ratio=0.2, n_ref=256, jsd_bins=20, seed=0):
    # W: (Nsamp, T, 4) -> (Nsamp, 4*len(scales))
    # method: EDM-Fuzzy | CMSE | FME | JSD-Fuzzy
    Nsamp, T, ns = W.shape
    F = np.empty((Nsamp, ns*len(scales)), dtype=float)

    method_key = method.strip().lower()
    def entropy_1d(x, seed_local):
        if method_key == 'edm-fuzzy':
            return edm_fuzzy_entropy_1d(x, scales=scales, m=m, r_ratio=r_ratio, n_ref=n_ref, seed=seed_local)
        if method_key == 'cmse':
            return cmse_1d(x, scales=scales, m=m, r_ratio=r_ratio)
        if method_key == 'fme':
            return fme_1d(x, scales=scales, m=m, r_ratio=r_ratio, n_ref=n_ref, seed=seed_local)
        if method_key == 'jsd-fuzzy':
            return jsd_fuzzy_entropy_1d(x, scales=scales, m=m, r_ratio=r_ratio, n_ref=n_ref, seed=seed_local, bins=jsd_bins)
        raise ValueError("Unknown method: %s" % method)

    for i in range(Nsamp):
        feats=[]
        for s in range(ns):
            ev = entropy_1d(W[i,:,s], seed_local=seed+1000*i+19*s)
            feats.append(ev)
        F[i] = np.concatenate(feats, axis=0)
    return F

# Utility logging untuk footprint komputasi
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")

def run_with_metrics(label, fn):
    # Catatan: footprint = estimasi peak memory Python via tracemalloc
    tracemalloc.start()
    t0 = time.perf_counter()
    c0 = time.process_time()
    result = fn()
    t1 = time.perf_counter()
    c1 = time.process_time()
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    metrics = {
        "wall_s": t1 - t0,
        "cpu_s": c1 - c0,
        "peak_mem_mb": peak / (1024*1024)
    }
    logging.info("%s | wall=%.2fs cpu=%.2fs peak_mem=%.2f MB", label, metrics["wall_s"], metrics["cpu_s"], metrics["peak_mem_mb"])
    return result, metrics

S = 10
scales = np.arange(1, S+1)
m = 2
r_ratio = 0.2
n_ref = 256
jsd_bins = 20

methods = ["EDM-Fuzzy", "CMSE", "FME", "JSD-Fuzzy"]
F_by_method = {}
F_metrics = {}
for name in methods:
    Fm, mtr = run_with_metrics(
        "Entropy %s" % name,
        lambda n=name: compute_features_entropy(W_s, scales=scales, method=n, m=m, r_ratio=r_ratio, n_ref=n_ref, jsd_bins=jsd_bins, seed=7)
    )
    # handle NaN jika ada (sinyal terlalu pendek) -> imputasi median per kolom
    Fdf = pd.DataFrame(Fm)
    Fdf = Fdf.fillna(Fdf.median(numeric_only=True))
    F_by_method[name] = Fdf.to_numpy()
    F_metrics[name] = mtr
    print(name, "feature shape:", F_by_method[name].shape)

# Tabel ringkas footprint komputasi fitur
pd.DataFrame(F_metrics).T

# Metode default untuk blok berikut (semua metode tetap dibandingkan di bawah).
default_method = "EDM-Fuzzy"
show_all_methods = True  # True = tampilkan semua metode di tabel/plot perbandingan
F = F_by_method[default_method]


In [ ]:
# Tampilkan footprint komputasi per metode (semua metode dihitung)
feature_metrics_table = pd.DataFrame(F_metrics).T
feature_metrics_table


# CV (Stabilitas Entropy)

`entropy_cv_report(F, S)`

**Input**
- `F`: `(Nsamp, 4*S)` fitur entropy gabungan

**Proses**
- reshape menjadi `E = F.reshape(Nsamp, 4, S)`
- hitung mean & std di axis sampel (Nsamp)
- CV per sensor per skala: `std / |mean|`
- `cv_avg`: rata-rata CV dari 4 sensor pada tiap skala

**Output**
- `cv_by_sensor`: `(4, S)`
- `cv_avg`: `(S,)` untuk plot stabilitas vs skala (semakin kecil → semakin stabil)


## 6) CV (Coefficient of Variation) untuk menunjukkan kestabilan entropy
CV dihitung **per skala** dari sebaran entropy (mis. across samples). Ini memudahkan menunjukkan apakah entropy stabil terhadap variasi sampel.

Catatan A: jika ingin menguji CV untuk beberapa pilihan `S`, jalankan loop pada beberapa `S`.
Catatan B: pada versi ini, CV dihitung **untuk semua metode** agar mudah dibandingkan.


In [ ]:
# Perbandingan kestabilan (CV) antar metode; semua metode ikut dibandingkan

def entropy_cv_report(F, S, nsensors=4):
    # F: (N, 4*S) -> CV per skala (averaged across sensors)
    N = F.shape[0]
    E = F.reshape(N, nsensors, S)  # (N,4,S)
    mean = E.mean(axis=0)          # (4,S)
    std  = E.std(axis=0, ddof=1)   # (4,S)
    cv   = std / (np.abs(mean)+1e-12)
    # report per sensor & averaged
    return cv, cv.mean(axis=0)

cv_by_method = {}
cv_avg_table = {}
for name, Fm in F_by_method.items():
    cv_by_sensor, cv_avg = entropy_cv_report(Fm, S=S)
    cv_by_method[name] = (cv_by_sensor, cv_avg)
    cv_avg_table[name] = cv_avg

# Plot perbandingan CV
plt.figure()
for name, cv_avg in cv_avg_table.items():
    plt.plot(np.arange(1,S+1), cv_avg, marker='o', label=name)
plt.xlabel("Scale s")
plt.ylabel("CV of entropy (avg sensors)")
plt.title("Stability of Entropy vs scale (per method)")
plt.legend()
plt.show()

# Default EDM–Fuzzy untuk kompatibilitas blok berikut
cv_by_sensor, cv_avg = cv_by_method["EDM-Fuzzy"]

cv_avg


In [ ]:
cv_table = pd.DataFrame(cv_avg_table)
cv_table.insert(0, "scale", np.arange(1, S+1))
cv_table


In [ ]:

# Recompute mean entropy per sensor (robust, order-independent)
def mean_entropy_table_for(F_local, S, nsensors=4):
    N = F_local.shape[0]
    E = F_local.reshape(N, nsensors, S)
    meanE = E.mean(axis=0)  # (nsensors,S)
    return pd.DataFrame(
        meanE,
        index=[f"Sensor {i}" for i in range(1, nsensors+1)],
        columns=[f"s={i}" for i in range(1, S+1)]
    )

if show_all_methods:
    mean_entropy_tables = {name: mean_entropy_table_for(Fm, S) for name, Fm in F_by_method.items()}
    mean_entropy_tables
else:
    mean_entropy_table = mean_entropy_table_for(F, S)
    mean_entropy_table


### Visualisasi entropy rata-rata per skala (opsional)
Menampilkan mean entropy untuk melihat trend vs skala.

In [ ]:

def plot_mean_entropy(F_local, S, title):
    N = F_local.shape[0]
    E = F_local.reshape(N, 4, S)
    meanE = E.mean(axis=0)  # (4,S)

    plt.figure()
    for i in range(4):
        plt.plot(np.arange(1,S+1), meanE[i], marker='o', label=f"Sensor {i+1}")
    plt.xlabel("Scale s")
    plt.ylabel("Mean entropy")
    plt.title(title)
    plt.legend()
    plt.show()

if show_all_methods:
    for name, Fm in F_by_method.items():
        plot_mean_entropy(Fm, S, f"Mean {name} Entropy per sensor")
else:
    plot_mean_entropy(F, S, f"Mean {default_method} Entropy per sensor")


# Klasifikasi Fault (ANN + Grid Search)

**Input**
- `F_by_method`: fitur entropy untuk semua metode
- `y_s`: label kelas (0=normal, 1..K skenario fault)

**Proses**
- Split data: `train_test_split(..., stratify=y)` atau `USE_TIME_SPLIT=True`
- Pipeline: `StandardScaler()` → `MLPClassifier()`
- GridSearch menyapu: `hidden_layer_sizes`, `alpha`, `learning_rate_init`, `activation`
- Jika `compare_methods = True`, model **dilatih untuk semua metode** dan dibandingkan

**Output**
- `train_metrics_by_method` + `train_metrics_table`: ringkasan accuracy & macro-F1 per metode
- Plot perbandingan performa antar metode
- `best` estimator terbaik per metode (diambil dari `GridSearchCV`)


## 7) Fault detection / classification dengan ANN
Input = gabungan vector entropy (4×S). Output = kelas fault (1..n) + normal (0).

Arsitektur hidden layer mengikuti best practice (Heaton):
- $H_1$ antara input dan output, sering dipakai: $\lfloor\frac{2}{3}I + O\rfloor$
- Alternatif: $H_1=I$, $H_2=\lfloor I/2 \rfloor$

Di sini dibuat beberapa kandidat dan dilakukan grid search.

In [ ]:

def eval_model(name, model, Xtr, ytr, Xte, yte):
    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    return {
        "model": name,
        "accuracy": float(accuracy_score(yte, pred)),
        "macro_f1": float(f1_score(yte, pred, average="macro"))
    }


In [ ]:

# Optional: gunakan split berbasis urutan (mengurangi leakage akibat overlap sampel)
USE_TIME_SPLIT = False  # set True untuk split berdasarkan urutan sampel

def time_split(F, y, test_frac=0.25):
    n=len(F); cut=int(np.floor((1-test_frac)*n))
    return F[:cut], F[cut:], y[:cut], y[cut:]


In [48]:
# train/test split + GridSearch ANN
compare_methods = True  # True = semua metode digunakan dan dibandingkan

# Ensure methods list exists when running this cell standalone
if "methods" not in globals():
    methods = ["EDM-Fuzzy", "CMSE", "FME", "JSD-Fuzzy"]

train_metrics_by_method = {}

def train_ann_for_F(F_local, label_name):
    # Return gs, split, pred untuk dipakai di blok berikut
    Xtr, Xte, ytr, yte = time_split(F_local, y_s, test_frac=0.25) if USE_TIME_SPLIT else train_test_split(F_local, y_s, test_size=0.25, random_state=42, stratify=y_s)

    I = F_local.shape[1]
    O = len(np.unique(y_s))
    H1_a = int(np.floor((2/3)*I + O))
    H1_b = I
    H1_c = min(2*I-1, int(0.9*I))
    H2_half = max(1, I//2)

    candidates = [
        (H1_a,),
        (H1_b,),
        (H1_a, H2_half),
        (H1_b, H2_half),
    ]

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPClassifier(max_iter=300, random_state=42))
    ])

    param_grid = {
        "mlp__hidden_layer_sizes": candidates,
        "mlp__alpha": [1e-4, 1e-3, 1e-2],
        "mlp__learning_rate_init": [1e-3, 3e-4],
        "mlp__activation": ["relu", "tanh"],
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    gs = GridSearchCV(pipe, param_grid=param_grid, cv=cv, n_jobs=-1, scoring="accuracy", verbose=1)
    gs.fit(Xtr, ytr)

    best = gs.best_estimator_
    pred = best.predict(Xte)

    metrics = {
        "best_params": gs.best_params_,
        "best_cv_acc": float(gs.best_score_),
        "test_acc": float(accuracy_score(yte, pred)),
        "macro_f1": float(f1_score(yte, pred, average="macro"))
    }
    return {"gs": gs, "Xtr": Xtr, "Xte": Xte, "ytr": ytr, "yte": yte, "pred": pred, "metrics": metrics, "label": label_name}

if compare_methods:
    train_results = {}
    for name in methods:
        res, mtr = run_with_metrics("ANN %s" % name, lambda n=name: train_ann_for_F(F_by_method[n], n))
        train_results[name] = res
        train_metrics_by_method[name] = {**res["metrics"], **mtr}

    # Ringkasan waktu/footprint training
    pd.DataFrame(train_metrics_by_method).T
else:
    res, mtr = run_with_metrics("ANN EDM-Fuzzy", lambda: train_ann_for_F(F, "EDM-Fuzzy"))
    train_metrics_by_method["EDM-Fuzzy"] = {**res["metrics"], **mtr}

# Ringkasan performa semua metode (tabel + plot)
train_metrics_table = pd.DataFrame(train_metrics_by_method).T
train_metrics_table

plt.figure()
plt.plot(train_metrics_table.index, train_metrics_table["test_acc"], marker='o', label="Test Acc")
plt.plot(train_metrics_table.index, train_metrics_table["macro_f1"], marker='o', label="Macro-F1")
plt.xlabel("Method")
plt.ylabel("Score")
plt.title("ANN performance by entropy method")
plt.legend()
plt.show()

# Default EDM–Fuzzy untuk kompatibilitas blok berikut
edm_res = train_results["EDM-Fuzzy"] if compare_methods else res
gs = edm_res["gs"]
Xtr, Xte, ytr, yte = edm_res["Xtr"], edm_res["Xte"], edm_res["ytr"], edm_res["yte"]
pred = edm_res["pred"]


Fitting 5 folds for each of 48 candidates, totalling 240 fits


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


Fitting 5 folds for each of 48 candidates, totalling 240 fits


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


Fitting 5 folds for each of 48 candidates, totalling 240 fits


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


Fitting 5 folds for each of 48 candidates, totalling 240 fits


In [49]:
# Display computing footprint per method during ANN training
train_metrics_table = pd.DataFrame(train_metrics_by_method).T
train_metrics_table


,best_params,best_cv_acc,test_acc,macro_f1,wall_s,cpu_s,peak_mem_mb
EDM-Fuzzy,"{'mlp__activation': 'relu', 'mlp__alpha': 0.00...",0.363983,0.354701,0.327599,380.527757,7.936433,1.849753
CMSE,"{'mlp__activation': 'tanh', 'mlp__alpha': 0.01...",0.341891,0.344017,0.309804,379.878244,10.103162,1.835746
FME,"{'mlp__activation': 'tanh', 'mlp__alpha': 0.00...",0.376078,0.361111,0.335166,378.005962,9.630845,1.833537
JSD-Fuzzy,"{'mlp__activation': 'relu', 'mlp__alpha': 0.00...",0.33046,0.307692,0.294167,377.11922,7.665416,1.830996


In [50]:
# Combined comparison table (feature + training metrics)
combined_metrics_table = pd.concat(
    [feature_metrics_table.add_prefix('feat_'), train_metrics_table.add_prefix('train_')],
    axis=1
)
combined_metrics_table


,feat_wall_s,feat_cpu_s,feat_peak_mem_mb,train_best_params,train_best_cv_acc,train_test_acc,train_macro_f1,train_wall_s,train_cpu_s,train_peak_mem_mb
EDM-Fuzzy,245.564343,241.526193,4.613760,"{'mlp__activation': 'relu', 'mlp__alpha': 0.00...",0.363983,0.354701,0.327599,380.527757,7.936433,1.849753
CMSE,5088.222125,5053.339517,0.628520,"{'mlp__activation': 'tanh', 'mlp__alpha': 0.01...",0.341891,0.344017,0.309804,379.878244,10.103162,1.835746
FME,474.409324,471.283683,6.577349,"{'mlp__activation': 'tanh', 'mlp__alpha': 0.00...",0.376078,0.361111,0.335166,378.005962,9.630845,1.833537
JSD-Fuzzy,344.156556,341.932342,5.716313,"{'mlp__activation': 'relu', 'mlp__alpha': 0.00...",0.33046,0.307692,0.294167,377.11922,7.665416,1.830996


In [51]:
# Quick stats preview for combined metrics
combined_metrics_table.describe()


,feat_wall_s,feat_cpu_s,feat_peak_mem_mb
count,4.000000,4.000000,4.000000
mean,1538.088087,1527.020434,4.383986
std,2368.611008,2352.759769,2.629465
min,245.564343,241.526193,0.628520
25%,319.508502,316.830805,3.617450
50%,409.282940,406.608013,5.165037
75%,1627.862524,1616.797642,5.931572
max,5088.222125,5053.339517,6.577349


In [52]:

best = gs.best_estimator_
pred = best.predict(Xte)

print(classification_report(yte, pred, labels=np.arange(len(scenario_names)), target_names=scenario_names))
print('Accuracy:', accuracy_score(yte, pred))
print('Macro-F1:', f1_score(yte, pred, average="macro"))
cm = confusion_matrix(yte, pred, labels=np.arange(len(scenario_names)))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=scenario_names)
disp.plot(xticks_rotation=45)
plt.show()


ValueError: Number of classes, 16, does not match size of target_names, 17. Try specifying the labels parameter

In [ ]:

# Baseline comparisons (trained on same split as ANN)
baseline_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, multi_class="auto"))
])

baseline_rf = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample"
)

results = []
# ANN best estimator
results.append({
    "model":"ANN (GridSearch best)",
    "accuracy": float(accuracy_score(yte, pred)),
    "macro_f1": float(f1_score(yte, pred, average="macro"))
})
results.append(eval_model("LogReg", baseline_lr, Xtr, ytr, Xte, yte))
results.append(eval_model("RandomForest", baseline_rf, Xtr, ytr, Xte, yte))

pd.DataFrame(results).sort_values(["macro_f1","accuracy"], ascending=False)


# Eksperimen Sensitivitas Jumlah Skala `S`

`run_experiment_S(...)`

**Tujuan:** melihat trade-off:
- Stabilitas entropy (CV) vs
- Performa klasifikasi (Accuracy/Macro-F1)
ketika `S` diubah.

**Proses per nilai S**
1) Hitung fitur `F` dengan ukuran `4*S`
2) Hitung `CV_mean` (rata-rata cv_avg)
3) Train MLP sederhana (tanpa grid besar) untuk estimasi performa
4) Simpan ringkasan ke tabel `exp`

**Output**
- `exp`: DataFrame berisi kolom `S, CV_mean, test_acc, macro_f1, I, H1`
- Plot CV vs S, Accuracy vs S, Macro-F1 vs S


## 8) Eksperimen: variasi jumlah skala `S` (untuk CV + akurasi)
Bagian ini mengulang pipeline fitur untuk beberapa `S` agar terlihat hubungan kestabilan (CV) dan performa klasifikasi.

Untuk speed: turunkan `MAX_PER_CLASS`, `n_ref`, atau `S_list`.

In [ ]:
def run_experiment_S(W, y, S_list=(5,8,10,12), m=2, r_ratio=0.2, n_ref=128, seed=7, use_time_split=False, method="EDM-Fuzzy"):
    rows=[]
    for S in S_list:
        scales=np.arange(1,S+1)
        F = compute_features_entropy(W, scales=scales, method=method, m=m, r_ratio=r_ratio, n_ref=n_ref, seed=seed)
        Fdf=pd.DataFrame(F).fillna(pd.DataFrame(F).median(numeric_only=True))
        F=Fdf.to_numpy()
        cv_by_sensor, cv_avg = entropy_cv_report(F, S=S)

        # simple MLP (no big grid) for quick trend
        Xtr, Xte, ytr, yte = time_split(F, y, test_frac=0.25) if use_time_split else train_test_split(F, y, test_size=0.25, random_state=42, stratify=y)
        I = F.shape[1]
        O = len(np.unique(y))
        H1 = int(np.floor((2/3)*I + O))

        clf = Pipeline([
            ("scaler", StandardScaler()),
            ("mlp", MLPClassifier(hidden_layer_sizes=(H1,), max_iter=200, random_state=42))
        ])
        clf.fit(Xtr, ytr)
        pred = clf.predict(Xte)
        rows.append({
            "S": S,
            "CV_mean": float(np.mean(cv_avg)),
            "test_acc": float(accuracy_score(yte, pred)),
            "macro_f1": float(f1_score(yte, pred, average="macro")),
            "I": I,
            "H1": H1,
            "method": method
        })

    exp = pd.DataFrame(rows)
    return exp


In [ ]:

# === Robust experiment computation (fully self-contained, no NameError) ===

if "S_list" not in globals():
    S_list = (5, 8, 10, 12)

if "exp" not in globals():
    exp = run_experiment_S(
        W_s, y_s,
        S_list=S_list,
        m=2,
        r_ratio=0.2,
        n_ref=128,
        seed=7,
        use_time_split=USE_TIME_SPLIT
    )

# ---- Plots ----
plt.figure()
plt.plot(exp["S"], exp["CV_mean"], marker="o")
plt.xlabel("S (max scale)")
plt.ylabel("Mean CV (entropy)")
plt.title("Entropy stability vs S")
plt.show()

plt.figure()
plt.plot(exp["S"], exp["test_acc"], marker="o")
plt.xlabel("S (max scale)")
plt.ylabel("Test accuracy")
plt.title("ANN accuracy vs S")
plt.show()

plt.figure()
plt.plot(exp["S"], exp["macro_f1"], marker="o")
plt.xlabel("S (max scale)")
plt.ylabel("Macro-F1")
plt.title("Macro-F1 vs S")
plt.show()


In [ ]:

plt.figure()
plt.plot(exp["S"], exp["macro_f1"], marker='o')
plt.xlabel("S (max scale)")
plt.ylabel("Macro-F1")
plt.title("Macro-F1 vs S")
plt.show()


In [ ]:
# Export tables to CSV (into metrics_exports/)
import os
os.makedirs('metrics_exports', exist_ok=True)
feature_metrics_table.to_csv('metrics_exports/feature_metrics_table.csv', index=True)
train_metrics_table.to_csv('metrics_exports/train_metrics_table.csv', index=True)
combined_metrics_table.to_csv('metrics_exports/combined_metrics_table.csv', index=True)
print('Wrote metrics_exports/feature_metrics_table.csv, metrics_exports/train_metrics_table.csv, metrics_exports/combined_metrics_table.csv')
